# Extreme Total Water Level Analysis under Sea Level Rise
### Coastal Risk Assessment — Palermo, Sicily (Mediterranean)

---

This notebook estimates **design extreme water levels** at a sandy beach site near Palermo, Italy, under present-day conditions and projected sea level rise (SLR) to 2075 using the SSP2-4.5 climate scenario.

The **Total Water Level (TWL)** at the shoreline is the sum of three components:

$$\text{TWL} = \eta_{tide} + \eta_{surge} + R_2$$

where $R_2$ is the 2% exceedance wave runup, which itself includes wave setup and swash.

Two empirical runup models are compared throughout:
- **Stockdon et al. (2006)** — the widely used formula separating incident and infragravity swash components
- **Atkinson et al. (2017)** — a simpler slope-dependent formulation

Extreme value analysis is then performed using two complementary approaches:
- **Annual Maxima (AM)** — fitted with a Generalised Extreme Value (GEV) distribution
- **Peaks Over Threshold (POT)** — fitted with a Generalised Pareto Distribution (GPD)

Return levels are estimated for return periods of 25, 50, 100, and 220 years, both under present conditions and with the 2075 SLR offset added.

---

**Key parameters:**
| Parameter | Value |
|---|---|
| Site | Palermo, Sicily |
| Beach slope ($\tan\beta$) | 0.098 |
| Correction factor | 1.11 |
| SLR 2075 (SSP2-4.5) | 0.41 m (interpolated) |
| Return periods | 25, 50, 100, 220 years |
| Extreme value methods | GEV (Annual Maxima), GPD (POT) |

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import genextreme, genpareto

# ── Plotting style ─────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Libraries loaded.")

## 2. Site Parameters

All key physical and scenario parameters are collected here so they are easy to modify for different sites or climate scenarios.

The **correction factor (COEF = 1.11)** accounts for the difference between offshore hindcast wave heights and the nearshore conditions at the study site — it scales both the significant wave height $H_{m0}$ and the storm surge.

The **representative slope** $\beta = \tan\beta \times \text{COEF}$ is the effective beach face slope used in the runup formulae. The beach slope at this site is 0.098 (moderately steep, typical of gravel-sand beaches in the Mediterranean).

**Sea Level Rise (SLR):** The SSP2-4.5 scenario projects a median SLR of 0.37 m by 2070 and 0.45 m by 2080 at Palermo. A linear interpolation gives 0.41 m for the 2075 design horizon.

In [ ]:
# ── Physical parameters ────────────────────────────────────────────
COEF      = 1.11          # nearshore correction factor (applied to Hm0 and surge)
SLOPE_RAW = 0.098         # measured beach face slope (tan β)
BETA      = SLOPE_RAW * COEF  # effective slope used in runup formulae

# ── Sea Level Rise: SSP2-4.5, Palermo ─────────────────────────────
SLR_2070 = 0.37           # median SLR at 2070 (m)
SLR_2080 = 0.45           # median SLR at 2080 (m)
SLR_2075 = SLR_2070 + (SLR_2080 - SLR_2070) * 0.5  # linear interpolation → 0.41 m

# ── Return periods of interest ─────────────────────────────────────
T_LIST = [25, 50, 100, 220]  # years

print(f"Effective beach slope (β):  {BETA:.4f}")
print(f"SLR at 2075 (SSP2-4.5):    {SLR_2075:.3f} m")

## 3. Load and Merge the Wave and Surge Records

Two time series are used:
- **Waves.csv** — hourly hindcast significant wave height ($H_{m0}$, m) and peak period ($T_p$, s)
- **surge.csv** — hourly tide ($\eta_{tide}$, m) and storm surge ($\eta_{surge}$, m)

Both records are merged on a common datetime index. Only timesteps present in both files are kept (`inner` join), ensuring a consistent dataset for the entire analysis.

> **Note:** Place `Waves.csv` and `surge.csv` in the same directory as this notebook before running.

In [ ]:
# ── Load raw CSV files ─────────────────────────────────────────────
waves = pd.read_csv("Waves.csv", low_memory=False)
surge = pd.read_csv("surge.csv", low_memory=False)

# Retain only the columns needed for this analysis
waves = waves[["year", "month", "day", "hour", "Hm0", "Tp"]]
surge = surge[["year", "month", "day", "hour", "surge", "tide"]]

# ── Build datetime index ───────────────────────────────────────────
for df_tmp, name in [(waves, "waves"), (surge, "surge")]:
    df_tmp["datetime"] = pd.to_datetime(
        df_tmp[["year", "month", "day", "hour"]]
    )
    df_tmp.set_index("datetime", inplace=True)

# ── Merge on datetime (inner join keeps only shared timesteps) ─────
df = waves.join(surge[["surge", "tide"]], how="inner")

print(f"Record length : {len(df):,} hourly timesteps")
print(f"Period covered: {df.index[0].date()} → {df.index[-1].date()}")
print(f"\nFirst five rows:")
df[["Hm0", "Tp", "surge", "tide"]].head()

## 4. Compute Total Water Level (TWL)

### 4.1 Apply the Nearshore Correction Factor

The correction factor is applied to both $H_{m0}$ and storm surge to account for the transformation of offshore conditions to the nearshore:

$$H_{m0,\text{corr}} = H_{m0} \times 1.11 \qquad \eta_{surge,\text{corr}} = \eta_{surge} \times 1.11$$

The deep-water wavelength is computed from linear wave theory:
$$L_0 = \frac{g T_p^2}{2\pi} \approx 1.56\,T_p^2$$

In [ ]:
df["Hm0_corr"]  = df["Hm0"]   * COEF
df["surge_corr"] = df["surge"] * COEF

# Deep-water wavelength from linear wave theory (L0 = g/2π * Tp²)
df["L0"] = 1.56 * df["Tp"] ** 2

print("Correction applied. Sample statistics:")
df[["Hm0", "Hm0_corr", "surge", "surge_corr", "L0"]].describe().round(3)

### 4.2 Wave Runup Models

Wave runup $R_2$ is the vertical height exceeded by 2% of wave runup events. Two models are applied:

**Stockdon et al. (2006):**  
This model separates runup into wave *setup* ($\bar{\eta}$) and *swash* ($S$) components, where swash is further split into incident-frequency ($S_{inc}$) and infragravity ($S_{ig}$) contributions:

$$\bar{\eta} = 0.35\,\beta\sqrt{H_{m0} L_0}$$
$$S = \sqrt{S_{inc}^2 + S_{ig}^2}, \quad S_{inc} = 0.75\,\beta\sqrt{H_{m0} L_0}, \quad S_{ig} = 0.06\sqrt{H_{m0} L_0}$$
$$R_2 = 1.1\left(\bar{\eta} + \frac{S}{2}\right)$$

**Atkinson et al. (2017):**  
A slightly simpler formulation that has shown good skill on Mediterranean-type beaches:

$$R_2 = 0.92\,\beta\sqrt{H_{m0} L_0} + 0.16\,H_{m0}$$

**Total Water Level:**
$$\text{TWL} = \eta_{tide} + \eta_{surge} + R_2$$

In [ ]:
# ── Stockdon et al. (2006) ─────────────────────────────────────────
df["setup_S"]  = 0.35 * BETA * np.sqrt(df["Hm0_corr"] * df["L0"])

S_inc = 0.75 * BETA * np.sqrt(df["Hm0_corr"] * df["L0"])
S_ig  = 0.06        * np.sqrt(df["Hm0_corr"] * df["L0"])
df["swash_S"] = np.sqrt(S_inc**2 + S_ig**2)

df["R2_stockdon"] = 1.1 * (df["setup_S"] + df["swash_S"] / 2.0)

# ── Atkinson et al. (2017) ─────────────────────────────────────────
df["R2_atkinson"] = 0.92 * BETA * np.sqrt(df["Hm0_corr"] * df["L0"]) + 0.16 * df["Hm0_corr"]

# ── Total Water Level ──────────────────────────────────────────────
df["TWL_stockdon"] = df["tide"] + df["surge_corr"] + df["R2_stockdon"]
df["TWL_atkinson"] = df["tide"] + df["surge_corr"] + df["R2_atkinson"]

print("TWL computed. Summary statistics (m):")
df[["R2_stockdon", "R2_atkinson", "TWL_stockdon", "TWL_atkinson"]].describe().round(3)

### 4.3 TWL Time Series — Quick Look

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df.index, df["TWL_atkinson"],
        linewidth=0.4, alpha=0.6, color="#2196F3", label="Atkinson (2017)")
ax.plot(df.index, df["TWL_stockdon"],
        linewidth=0.4, alpha=0.6, color="#E91E63", label="Stockdon (2006)")

ax.set_xlabel("Year")
ax.set_ylabel("Total Water Level (m)")
ax.set_title("Hourly TWL Time Series — Palermo site")
ax.legend(loc="upper right")
fig.tight_layout()
plt.savefig("TWL_timeseries.png", dpi=150)
plt.show()

## 5. Annual Maxima Method — GEV Distribution

The **Annual Maxima (AM)** method extracts the single largest TWL value from each calendar year. These $n$ annual maxima are then fitted to a **Generalised Extreme Value (GEV)** distribution, justified by the Fisher-Tippett-Gnedenko theorem:

$$F(x) = \exp\!\left[-\left(1 + \xi\,\frac{x - \mu}{\sigma}\right)^{-1/\xi}\right]$$

where $\mu$ (location), $\sigma$ (scale $> 0$), and $\xi$ (shape) are estimated by maximum likelihood. The **return level** $z_T$ for return period $T$ is:

$$z_T = \mu - \frac{\sigma}{\xi}\left[1 - (-\ln(1-1/T))^{-\xi}\right]$$

> *Note:* `scipy.stats.genextreme` uses the convention $c = -\xi$, so a positive `c` in SciPy corresponds to a **Weibull** (bounded upper tail) shape, while a negative `c` gives a **Fréchet** (heavy) tail.

In [ ]:
def fit_gev_am(series, T_list, slr):
    """
    Fit a GEV distribution to annual maxima of `series`.

    Parameters
    ----------
    series : pd.Series  — hourly TWL time series with DatetimeIndex
    T_list : list       — return periods (years)
    slr    : float      — sea level rise offset (m) added to get future RLs

    Returns
    -------
    rl_now  : dict  {T: return_level_present}
    rl_slr  : dict  {T: return_level_2075}
    am      : pd.Series  — annual maxima values
    am_time : pd.Series  — timestamps of those maxima
    params  : tuple (c, loc, scale)
    """
    # Extract one maximum per calendar year
    annual = series.resample("AS")
    am      = annual.max().dropna()
    am_time = annual.apply(lambda x: x.idxmax()).dropna()

    # Fit GEV by maximum likelihood
    c, loc, scale = genextreme.fit(am)

    print(f"  Years of record : {len(am)}")
    print(f"  Mean annual max : {am.mean():.3f} m")
    print(f"  GEV  c (= -ξ)  : {c:.4f}   loc : {loc:.4f}   scale : {scale:.4f}")

    gev = genextreme(c, loc, scale)
    rl_now = {T: gev.ppf(1 - 1.0 / T) for T in T_list}
    rl_slr = {T: v + slr              for T, v in rl_now.items()}

    return rl_now, rl_slr, am, am_time, (c, loc, scale)

In [ ]:
print("=" * 55)
print("GEV fit — Atkinson TWL")
print("=" * 55)
rl_now_A_gev, rl_slr_A_gev, am_A, am_time_A, gev_params_A = fit_gev_am(
    df["TWL_atkinson"], T_LIST, SLR_2075
)

print()
print("=" * 55)
print("GEV fit — Stockdon TWL")
print("=" * 55)
rl_now_S_gev, rl_slr_S_gev, am_S, am_time_S, gev_params_S = fit_gev_am(
    df["TWL_stockdon"], T_LIST, SLR_2075
)

In [ ]:
# ── Print GEV return levels ────────────────────────────────────────
headers = ["T (yr)", "Atkinson — Present", "Atkinson — 2075",
                      "Stockdon — Present", "Stockdon — 2075"]
print(f"{'T (yr)':>7}  {'Atk Present':>13}  {'Atk 2075':>10}  "
      f"{'Stk Present':>13}  {'Stk 2075':>10}")
print("-" * 62)
for T in T_LIST:
    print(f"{T:>7}  {rl_now_A_gev[T]:>13.3f}  {rl_slr_A_gev[T]:>10.3f}  "
          f"{rl_now_S_gev[T]:>13.3f}  {rl_slr_S_gev[T]:>10.3f}")
print(f"\nAll values in metres  |  SLR 2075 = {SLR_2075:.3f} m")

### 5.1 GEV Diagnostics — Annual Maxima and Distribution Fit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, am, params, label, color in zip(
    axes,
    [am_A, am_S],
    [gev_params_A, gev_params_S],
    ["Atkinson", "Stockdon"],
    ["#2196F3", "#E91E63"],
):
    c, loc, scale = params
    ax.hist(am, bins=9, density=True, alpha=0.55, color=color,
            edgecolor="white", linewidth=0.6, label="Annual maxima")

    x = np.linspace(am.min() - 0.3, am.max() + 0.8, 200)
    ax.plot(x, genextreme.pdf(x, c, loc, scale), "k-",
            linewidth=2, label=f"GEV fit (ξ={-c:.3f})")

    ax.set_title(f"GEV Fit — {label}")
    ax.set_xlabel("Annual Maximum TWL (m)")
    ax.set_ylabel("Probability Density")
    ax.legend()

fig.tight_layout()
plt.savefig("GEV_fit_diagnostics.png", dpi=150)
plt.show()

### 5.2 Time Series with Annual Maxima Highlighted

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))

ax.plot(df.index, df["TWL_atkinson"],
        linewidth=0.4, alpha=0.45, color="#2196F3", label="TWL — Atkinson")
ax.plot(df.index, df["TWL_stockdon"],
        linewidth=0.4, alpha=0.45, color="#E91E63", label="TWL — Stockdon")

ax.scatter(am_time_A.values, am_A.values,
           color="#0D47A1", s=30, zorder=5, label="Annual max — Atkinson")
ax.scatter(am_time_S.values, am_S.values,
           facecolors="none", edgecolors="#880E4F", s=45,
           linewidths=1.2, zorder=5, label="Annual max — Stockdon")

ax.set_xlabel("Year")
ax.set_ylabel("Total Water Level (m)")
ax.set_title("TWL Time Series with Annual Maxima — Palermo site")
ax.legend(ncol=2, fontsize=9)
fig.tight_layout()
plt.savefig("TWL_annual_maxima.png", dpi=150)
plt.show()

## 6. Peaks-Over-Threshold Method — GPD

The **Peaks-Over-Threshold (POT)** method makes more efficient use of the data by retaining all independent exceedances above a high threshold $u$, not just one value per year. By the Pickands-Balkema-de Haan theorem, these excesses $x - u$ asymptotically follow a **Generalised Pareto Distribution (GPD)**:

$$F(y) = 1 - \left(1 + \xi\,\frac{y}{\sigma}\right)^{-1/\xi}, \quad y = x - u > 0$$

The threshold is set at the **98th percentile** of the TWL series. Because storms produce clusters of consecutive exceedances, a **declustering** step is applied: consecutive exceedances within 48 hours are grouped, and only the peak of each group is retained as an independent storm event.

The return level for return period $T$ (in years) is:
$$z_T = u + \frac{\sigma}{\xi}\left[(T\lambda)^\xi - 1\right]$$

where $\lambda$ is the mean number of independent storm peaks per year.

In [ ]:
# ── Declustering: find independent storm peaks ─────────────────────
def decluster_peaks(series, threshold, min_gap_hours=48):
    """
    Identify independent storm peaks above `threshold`.

    Algorithm
    ---------
    1. Flag all hourly timesteps exceeding the threshold.
    2. Consecutive exceedances closer than `min_gap_hours` belong
       to the same storm cluster.
    3. Retain only the maximum within each cluster (the peak).

    Returns
    -------
    pd.Series of independent peak values indexed by their timestamps.
    """
    over     = series[series > threshold]
    if over.empty:
        return pd.Series(dtype=float)

    # Time gaps between consecutive exceedances (hours)
    gaps = over.index.to_series().diff().dt.total_seconds().div(3600)

    # A new cluster starts wherever the gap exceeds the separation window
    new_cluster = np.where(gaps > min_gap_hours)[0]
    starts      = np.insert(new_cluster, 0, 0)       # first point always starts a cluster

    peaks = {}
    for i, s in enumerate(starts):
        e = starts[i + 1] if i + 1 < len(starts) else len(over)
        cluster = over.iloc[s:e]
        peaks[cluster.idxmax()] = cluster.max()

    return pd.Series(peaks).sort_index()

In [ ]:
# ── POT fitting function ───────────────────────────────────────────
def fit_pot_gpd(series, thresh_quantile, min_gap_hrs, T_list, slr):
    """
    Fit a GPD to declustered storm peaks above a quantile threshold.

    Returns
    -------
    rl_now  : dict  {T: present return level}
    rl_slr  : dict  {T: future return level}
    peaks   : pd.Series  — independent storm peak values
    u       : float      — threshold value
    params  : tuple (shape ξ, scale σ)
    """
    u       = series.quantile(thresh_quantile)
    peaks   = decluster_peaks(series, u, min_gap_hrs)
    excesses = peaks - u

    # Record length in years
    n_years   = (series.index[-1] - series.index[0]).total_seconds() / (365.25 * 24 * 3600)
    lambda_yr = len(peaks) / n_years   # mean storms per year

    print(f"  Threshold u         : {u:.3f} m  (q = {thresh_quantile})")     
    print(f"  Raw exceedances     : {(series > u).sum():,}  hourly points")
    print(f"  Independent peaks   : {len(peaks)}  (after {min_gap_hrs} h declustering)")
    print(f"  Annual storm rate λ : {lambda_yr:.2f} events / year")

    # Fit GPD — location fixed at 0 (excess by definition starts at 0)
    xi, _, sigma = genpareto.fit(excesses, floc=0)

    print(f"  GPD shape ξ         : {xi:.4f}")
    print(f"  GPD scale σ         : {sigma:.4f}")

    def return_level(T):
        if abs(xi) < 1e-9:           # exponential tail (xi → 0)
            return u + sigma * np.log(T * lambda_yr)
        return u + (sigma / xi) * ((T * lambda_yr) ** xi - 1.0)

    rl_now = {T: return_level(T)       for T in T_list}
    rl_slr = {T: return_level(T) + slr for T in T_list}

    return rl_now, rl_slr, peaks, u, (xi, sigma)

In [ ]:
THRESH_Q    = 0.98   # 98th-percentile threshold
MIN_GAP_HRS = 48     # minimum separation between independent storms (hours)

print("=" * 55)
print("GPD fit — Atkinson TWL")
print("=" * 55)
rl_now_A_pot, rl_slr_A_pot, peaks_A, u_A, gpd_params_A = fit_pot_gpd(
    df["TWL_atkinson"], THRESH_Q, MIN_GAP_HRS, T_LIST, SLR_2075
)

print()
print("=" * 55)
print("GPD fit — Stockdon TWL")
print("=" * 55)
rl_now_S_pot, rl_slr_S_pot, peaks_S, u_S, gpd_params_S = fit_pot_gpd(
    df["TWL_stockdon"], THRESH_Q, MIN_GAP_HRS, T_LIST, SLR_2075
)

In [ ]:
# ── Print POT return levels ────────────────────────────────────────
print(f"{'T (yr)':>7}  {'Atk Present':>13}  {'Atk 2075':>10}  "
      f"{'Stk Present':>13}  {'Stk 2075':>10}")
print("-" * 62)
for T in T_LIST:
    print(f"{T:>7}  {rl_now_A_pot[T]:>13.3f}  {rl_slr_A_pot[T]:>10.3f}  "
          f"{rl_now_S_pot[T]:>13.3f}  {rl_slr_S_pot[T]:>10.3f}")
print(f"\nAll values in metres  |  SLR 2075 = {SLR_2075:.3f} m")

### 6.1 POT Diagnostics — Time Series with Peaks and Threshold

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))

# Continuous TWL series
ax.plot(df.index, df["TWL_atkinson"],
        lw=0.4, alpha=0.4, color="#2196F3", label="TWL — Atkinson")
ax.plot(df.index, df["TWL_stockdon"],
        lw=0.4, alpha=0.4, color="#E91E63", label="TWL — Stockdon")

# Threshold lines
ax.axhline(u_A, color="#2196F3", lw=1.2, ls="--", alpha=0.8,
           label=f"Threshold A = {u_A:.2f} m (q98)")
ax.axhline(u_S, color="#E91E63", lw=1.2, ls="--", alpha=0.8,
           label=f"Threshold S = {u_S:.2f} m (q98)")

# Declustered peaks
ax.scatter(peaks_A.index, peaks_A,
           color="#0D47A1", s=30, marker="^", zorder=5,
           label=f"Storm peaks — Atkinson (n={len(peaks_A)})")
ax.scatter(peaks_S.index, peaks_S,
           facecolors="none", edgecolors="#880E4F", s=40,
           linewidths=1.2, zorder=5,
           label=f"Storm peaks — Stockdon (n={len(peaks_S)})")

ax.set_xlabel("Year")
ax.set_ylabel("Total Water Level (m)")
ax.set_title("POT Analysis — Declustered Storm Peaks above 98th Percentile Threshold")
ax.legend(ncol=3, fontsize=8.5)
fig.tight_layout()
plt.savefig("POT_peaks_timeseries.png", dpi=150)
plt.show()

### 6.2 GPD Fit — Excess Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, peaks, u, params, label, color in zip(
    axes,
    [peaks_A, peaks_S],
    [u_A, u_S],
    [gpd_params_A, gpd_params_S],
    ["Atkinson", "Stockdon"],
    ["#2196F3", "#E91E63"],
):
    xi, sigma = params
    excesses  = peaks - u

    ax.hist(excesses, bins=12, density=True, alpha=0.55, color=color,
            edgecolor="white", lw=0.6, label="Observed excesses")

    x = np.linspace(0, excesses.max() * 1.15, 200)
    ax.plot(x, genpareto.pdf(x, xi, loc=0, scale=sigma),
            "k-", lw=2, label=f"GPD fit (ξ={xi:.3f})")

    ax.set_title(f"GPD Fit — {label}  (u = {u:.2f} m)")
    ax.set_xlabel("Excess above Threshold (m)")
    ax.set_ylabel("Probability Density")
    ax.legend()

fig.tight_layout()
plt.savefig("GPD_fit_diagnostics.png", dpi=150)
plt.show()

## 7. Return Level Plots — Both Methods Compared

The return level plot shows how the design TWL grows with return period, and how the two extreme value methods (GEV and GPD) compare. Dashed lines include the 2075 SLR offset. A consistent gap between present and future curves equal to the SLR offset (0.41 m) is expected.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

T_arr = np.array(T_LIST, dtype=float)

titles   = ["Annual Maxima (GEV)", "Peaks-Over-Threshold (GPD)"]
rl_sets  = [
    (rl_now_A_gev, rl_slr_A_gev, rl_now_S_gev, rl_slr_S_gev),
    (rl_now_A_pot, rl_slr_A_pot, rl_now_S_pot, rl_slr_S_pot),
]

for ax, title, (now_A, slr_A, now_S, slr_S) in zip(axes, titles, rl_sets):

    ax.plot(T_arr, [now_A[T] for T in T_LIST], "o-",
            color="#2196F3", lw=1.8, label="Present — Atkinson")
    ax.plot(T_arr, [slr_A[T] for T in T_LIST], "o--",
            color="#2196F3", lw=1.4, alpha=0.7, label="2075 — Atkinson")

    ax.plot(T_arr, [now_S[T] for T in T_LIST], "s-",
            color="#E91E63", lw=1.8, label="Present — Stockdon")
    ax.plot(T_arr, [slr_S[T] for T in T_LIST], "s--",
            color="#E91E63", lw=1.4, alpha=0.7, label="2075 — Stockdon")

    ax.set_xscale("log")
    ax.set_xticks(T_LIST)
    ax.set_xticklabels(T_LIST)
    ax.set_xlabel("Return Period T (years)")
    ax.set_title(title)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Extreme Water Level (m)")

fig.suptitle("Return Level Curves — Palermo (SSP2-4.5, SLR 2075 = 0.41 m)",
             fontsize=12, y=1.01)
fig.tight_layout()
plt.savefig("Return_levels_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Summary Table — Design Return Levels

The table below consolidates all return level estimates. Values represent the extreme TWL (in metres above mean sea level) expected to be exceeded on average once in $T$ years.

In [ ]:
rows = []
for T in T_LIST:
    rows.append({
        "Return Period (yr)": T,
        "GEV — Atkinson (present, m)": round(rl_now_A_gev[T], 3),
        "GEV — Atkinson (2075, m)"  : round(rl_slr_A_gev[T], 3),
        "GEV — Stockdon (present, m)": round(rl_now_S_gev[T], 3),
        "GEV — Stockdon (2075, m)"  : round(rl_slr_S_gev[T], 3),
        "GPD — Atkinson (present, m)": round(rl_now_A_pot[T], 3),
        "GPD — Atkinson (2075, m)"  : round(rl_slr_A_pot[T], 3),
        "GPD — Stockdon (present, m)": round(rl_now_S_pot[T], 3),
        "GPD — Stockdon (2075, m)"  : round(rl_slr_S_pot[T], 3),
    })

summary = pd.DataFrame(rows).set_index("Return Period (yr)")
summary

In [ ]:
summary.to_csv("return_levels_summary.csv")
print("Summary saved to return_levels_summary.csv")

## 9. Discussion

**Agreement between methods:**  
The GEV (Annual Maxima) and GPD (POT) approaches generally produce consistent return levels when the record is long enough and the threshold is well-chosen. Minor divergence at longer return periods is expected — the GEV extrapolates the tail using only $n$ annual maxima, while the GPD uses all independent storm peaks above the threshold, giving it more information about the tail behaviour.

**Runup model comparison:**  
The Stockdon (2006) formula tends to give slightly different runup magnitudes than Atkinson (2017) because of how infragravity swash is handled. On moderately steep beaches like this site ($\tan\beta \approx 0.1$), the two models are generally in reasonable agreement. At gentler slopes, infragravity energy (captured by Stockdon's $S_{ig}$ term) would begin to dominate and the divergence would grow.

**Sea level rise impact:**  
The 0.41 m SLR offset under SSP2-4.5 by 2075 shifts all return level curves upward by a constant amount. This means a flood event that currently has a 1-in-100 year probability could occur considerably more frequently by mid-century — a shift that has direct implications for coastal defence design.

**Limitations:**  
- The response method (adding wave runup to the still water level) assumes statistical independence between waves and storm surge; in reality, severe storms tend to produce both simultaneously.
- No uncertainty bounds (confidence intervals) are shown on the return levels; bootstrapping the GEV/GPD fits would be the natural next step.
- SLR is taken as a deterministic offset; a probabilistic SLR ensemble would better capture projection uncertainty.

---

**References**

- Stockdon, H.F., Holman, R.A., Howd, P.A., Sallenger, A.H. (2006). Empirical parameterization of setup, swash, and runup. *Coastal Engineering*, 53(7), 573–588.
- Atkinson, A.L., Power, H.E., Moura, T., Hammond, T., Callaghan, D.P., Baldock, T.E. (2017). Assessment of runup predictions by empirical models on non-planar beaches. *Coastal Engineering*, 119, 15–31.
- Coles, S. (2001). *An Introduction to Statistical Modeling of Extreme Values*. Springer, London.
- IPCC AR6 (2021). *Sea Level Projections, Chapter 9 — Ocean, Cryosphere and Sea Level Change.*